In [ ]:
!pip install -q langchain langchain-openai openai tiktoken chromadb langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.0/603.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 5.6 MB/s eta 0:00:0

In [ ]:
!wget https://github.com/chatgpt-kr/chatgpt-api-tutorial/raw/main/ch05/data.zip

--2024-10-14 05:11:04--  https://github.com/chatgpt-kr/chatgpt-api-tutorial/raw/main/ch05/data.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chatgpt-kr/chatgpt-api-tutorial/main/ch05/data.zip [following]
--2024-10-14 05:11:04--  https://raw.githubusercontent.com/chatgpt-kr/chatgpt-api-tutorial/main/ch05/data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 835816 (816K) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>] 816.23K  --.-KB/s    in 0.06s   

2024-10-14 05:11:04 (14.2 MB/s) - ‘data.zip’ saved [835816/835816]



In [ ]:
!unzip data

Archive:  data.zip
  inflating: 1.txt                   
  inflating: 10.txt                  
  inflating: 11.txt                  
  inflating: 12.txt                  
  inflating: 13.txt                  
  inflating: 14.txt                  
  inflating: 15.txt                  
  inflating: 16.txt                  
  inflating: 17.txt                  
  inflating: 18.txt                  
  inflating: 19.txt                  
  inflating: 2.txt                   
  inflating: 20.txt                  
  inflating: 21.txt                  
  inflating: 22.txt                  
  inflating: 23.txt                  
  inflating: 24.txt                  
  inflating: 25.txt                  
  inflating: 26.txt                  
  inflating: 27.txt                  
  inflating: 28.txt                  
  inflating: 29.txt                  
  inflating: 3.txt                   
  inflating: 30.txt                  
  inflating: 31.txt                  
  inflating: 32.txt            

# Langchain 세팅

In [2]:
import os

OPENAI_API_KEY = "~~~~~~~~~~~~~~~~~~~~"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.document_loaders import DirectoryLoader
from collections import Counter

# 다수의 문서 불러오기
DirectoryLoader() 함수를 이용해 특정 디렉토리 내의 모든 문서를 로딩할 수 있다.

In [ ]:
# glob 옵션을 이용해 어떤 확장자를 가진 파일을 로딩할지 결정
loader = DirectoryLoader(".", glob="*.txt", loader_cls=TextLoader)
documents = loader.load()

len(documents)

57

In [ ]:
documents[1]

Document(metadata={'source': '16.txt'}, page_content='정책 제목: 독립유공자후손장학금\n서울시 교육정책과에서 주관하는 교육 정책으로, 독립유공자 4대~6대 후손을 지원하기 위한 장학금이 제공됩니다. 이 정책은 2023년 3월 1일부터 2023년 12월 31일까지 진행됩니다. 장학금은 매년 300만원이 지급되며, 총 120명에게 지원될 예정입니다.\n\n신청 자격은 18세부터 100세까지의 나이를 가진 독립유공자 4대~6대 후손으로, 서울 소재 대학에 재학 중인 학생이나 서울 시민(자녀 포함)이 비서울 소재 대학에 재학 중인 경우에 해당합니다. 학력은 대학 재학자여야 하며, 전공이나 취업 상태, 특화 분야에 대한 요건은 없습니다. 다만 초과학기 학생은 신청이 제한됩니다.\n\n신청은 홈페이지 또는 우편을 통해 진행되며, 서류심사를 거친 후 최종 선발 결과가 발표됩니다. 제출해야 할 서류는 신청서, 재(휴)학증명서, 독립유공자 후손 증빙서류, 성적증명서, 경제상황 증빙서류입니다. 이 정책은 서울장학재단에서 운영되며, 자세한 내용은 https://www.hissf.or.kr에서 확인하실 수 있습니다.')

In [ ]:
documents[21]

Document(metadata={'source': '3.txt'}, page_content='정책 이름: 서울 청년 밀키트 창업지원 사업\n사회적 기업과 서울시청이 협력하여, 민간과 공공의 보유한 창업 인프라를 활용하여 청년들을 대상으로 밀키트 교육을 제공하는 프로그램이 진행됩니다. 이 프로그램은 판로 개척과 일자리 창출에 기여하기 위해 밀키트 창업교육, 시제품 개발 및 컨설팅을 지원합니다. 사업 운영 기간은 2023년 3월 1일부터 2023년 11월 30일까지이며, 신청 기간은 2023년 3월 1일부터 2023년 9월 30일까지입니다. 총 45개 팀(3개 기수, 기수별 15개 팀)이 지원을 받을 수 있습니다. 신청자격은 19세에서 39세까지의 청년으로, 공고일 기준으로 서울에 거주하고 창업 희망자여야 합니다. 학력, 전공, 취업 상태, 특화 분야에 대한 요건은 없습니다. 신청 방법과 신청서류는 관련 공고문을 확인하시면 자세히 안내되어 있습니다. 이 프로그램은 서울시농수산식품공사에서 운영됩니다.')

# 청킹

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = text_splitter.split_documents(documents)
len(texts)

In [ ]:
# 어떤 문서가 분할 되었는지 확인
texts[0]

Document(metadata={'source': '53.txt'}, page_content='정책 제목: 서울시 취업날개서비스 운영\n정책 유형은 "일자리"이며, 이 정책은 서울시청 일자리정책과에서 주관하고 있습니다. 해당 정책은 구직과정에서 경제적인 부담을 겪고 있는 청년들을 지원하기 위해 면접용 정장을 무료로 대여하는 "취업날개서비스"를 추진하고 있습니다. 지원 내용으로는 면접용 정장의 무료 대여가 포함되어 있습니다. 이 정책은 2023년 1월 1일부터 2023년 12월 31일까지 사업 운영기간이며, 신청은 2023년 1월 1일부터 2023년 12월 31일까지 가능합니다. 지원 규모는 총 48,000명입니다. 관련 사이트는 "https://job.seoul.go.kr/www/add_service/openChothesChest.do?method=selectOpenChothesChest"에서 확인하실 수 있습니다. 신청 자격은 18세부터 39세까지이며, 서울에 거주하거나 서울 소재 학교 재학생 또는 졸업생으로서 서울에 거소를 둔 청년들이 대상입니다. 학력, 전공, 취업 상태, 특화 분야에는 제한이 없습니다. 신청은 "https://job.seoul.go.kr/www/add_service/openChothesChest.do?method=selectOpenChothesChest"에서 온라인으로 진행하며, 심사 및 발표는 대여 가능한 장소에서 이루어집니다. 제출해야 하는 서류에는 서울 거주 사실이나 서울 소재 학교 재학생/졸업생임을 증명할 수 있는 서류와 면접을 보는 회사의 면접 확인 서류 등이 포함됩니다. 대여는 연간 최대 10회까지 가능하며, 대여기간은 3박 4일입니다. 추가적인 사항은 해당 링크에서 확인하실 수 있습니다.')

In [ ]:
source_lst = []
for i in range(0, len(texts)):
  source_lst.append(texts[i].metadata['source'])

element_counts = Counter(source_lst)
filtered_counts = {key: value for key, value in element_counts.items() if value >= 2}
print('2개 이상으로 분할된 문서 :', filtered_counts)
print('분할된 텍스트의 개수 :', len(documents) + len(filtered_counts))

2개 이상으로 분할된 문서 : {'48.txt': 2, '49.txt': 2, '36.txt': 2, '23.txt': 2, '22.txt': 2, '31.txt': 2, '40.txt': 2}
분할된 텍스트의 개수 : 64


# VectorDB 구성

In [ ]:
embedding = OpenAIEmbeddings()
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embedding
)

vectordb

<ipython-input-12-a02ae872009b>:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings()


# 검색기(Retriever) 생성

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
query = "신혼 부부를 위한 정책이 있어?"
docs = retriever.get_relevant_documents(query)

<ipython-input-15-55e015aa7339>:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


In [ ]:
len(docs)

4

In [ ]:
docs[0]

Document(metadata={'source': '23.txt'}, page_content='이 정책은 주거마련에 대한 부담을 완화하여 혼인수 감소와 출산기피 현상을 해결하고, 더 나은 주거환경을 제공하기 위해 서울시에서 운영하는 정책입니다. 대상 가구는 총 8,000가구로 제한되며, 지원 기간은 2023년 1월 1일부터 2023년 12월 31일까지입니다.\n\n이 정책에 참여하기 위해서는 다음의 신청자격을 충족해야 합니다. 먼저, 서울시민이거나 대출 후 1개월 이내에 서울로 전입 예정이어야 합니다. 또한 혼인신고일 기준으로 7년 이내의 신혼부부이거나 서울시 추천서 신청일로부터 6개월 이내에 결혼식 예정인 예비신혼부부여야 합니다. 부부의 합산 연소득은 9천 7백만원 이하여야 하며, 본인 및 배우자는 무주택자여야 합니다. 또한, 특정 주택 조건을 충족하는 주택의 임대차계약을 체결한 자에게 대출이 지원됩니다.\n\n이 정책은 서울주거포털(https://housing.seoul.go.kr)을 통해 온라인으로 신청할 수 있습니다. 필요한 제출서류로는 주민등록등본, 가족관계증명서, 혼인관계증명서, 그리고 임대차계약서가 있습니다.\n\n이 정책은 서울시청 주택정책과에서 운영되며, 자세한 사항은 해당 사이트(https://housing.seoul.go.kr)에서 확인하실 수 있습니다.')

In [ ]:
# 문서의 출처 확인

for doc in docs:
    print(doc.metadata['source'])

23.txt
23.txt
40.txt
39.txt


각각의 문서가 명확하게 다른 주제에 대해서 이야기 한다면 overlap을 조금 많이 줘도 괜찮을 것이다.

In [ ]:
docs[1]

Document(metadata={'source': '23.txt'}, page_content='정책제목: 신혼부부 임차보증금 지원\n정부에서는 주거 관련 정책을 통해 부담을 완화하여 더 나은 주거환경을 제공하고자 합니다. 현재 주거마련에 대한 부담으로 인해 혼인수가 감소하고 출산기피 현상이 발생하고 있습니다. 따라서 주거비 부담을 완화하여 이러한 문제를 해결하고, 좋은 주거환경을 제공하고자 합니다.\n\n이 정책은 서울시청 주택정책과에서 주관하며, 주거 마련에 대한 부담을 완화하기 위한 내용을 포함하고 있습니다. 지원 대상은 관내 임차보증금 7억 이내의 주택 또는 주거용 오피스텔에 대해 해당하는 서울시민이나 서울로 전입 예정인 자입니다. 대출한도는 임차보증금의 90% 이내 또는 2억원 중 작은 금액이며, 대출금의 최대 연 3.6% 이차보전 및 최장 10년까지 지원됩니다.\n\n주택조건과 대출형식은 한국주택금융공사 보증 및 협약은행(국민, 하나, 신한) 대출, 그리고 서울시 이차보전이 적용됩니다. 이 정책은 2023년 1월 1일부터 2023년 12월 31일까지 운영되며, 신청자격은 1세부터 100세까지의 연령을 가진 사람들이 해당합니다.\n\n대출을 받기 위한 추가 요건으로는 혼인신고일 기준으로 7년 이내의 신혼부부이거나 서울시 추천서 신청일로부터 6개월 이내에 결혼식 예정인 예비신혼부부여야 합니다. 또한 부부합산 연소득이 9천 7백만원 이하이고 본인 및 배우자가 무주택자여야 합니다. 대출을 받을 주택은 특정 조건을 충족하는 주택의 임대차계약을 체결한 사람들을 대상으로 합니다.\n\n이 정책은 서울주거포털(https://housing.seoul.go.kr)에서 온라인으로 신청할 수 있습니다. 필요한 서류로는 주민등록등본, 가족관계증명서, 혼인관계증명서, 그리고 임대차계약서가 제출되어야 합니다.\n\n이 정책은 주거마련에 대한 부담을 완화하여 혼인수 감소와 출산기피 현상을 해결하고, 더 나은 주거환경을 제공하기 위해 서울시에서 운영하는 정책입니다. 대상 가구는 

### k개의 결과 반환
질문에 대해 답변이 들어있는 문서는 청크를 모두 고려했을 때 최대 개수가 2개임을 위에서 확인.

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

# Langchain 모델 구축
RetrievalQA.from_chain_type()을 이용해 llm 모델 기반의 QA 챗봇을 구축합니다. 이번 실습에서는 프롬프트를 따로 만들지 않고 `stuff`로 설정하여 기본 프롬프트를 이용한 QA 챗봇을 만들어 봅니다.

참고로 `stuff`로 설정했을 때 기본 프롬프트는 다음과 같습니다.
```
Use the following pieces of context to answer the users question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
{텍스트}

{질문}
```

`{텍스트}`에는 사용자의 질문으로부터 높은 유사도를 가진 텍스트가 들어가게 됩니다. `{질문}` 부분은 사용자의 질문이 들어가게 됩니다.

`return_source_documents`는 챗봇의 답변에 사용된 텍스트들의 출처를 표시할 것인지를 의미하게 됩니다.

In [ ]:
# 기본적으로 주어지는 QA Prompt를 사용할 예정(stuff)
qa_chain = RetrievalQA.from_chain_type(
    llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0),
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# 질의하기

In [ ]:
input_text = "대출과 관련된 정책이 궁금합니다."
chatbot_response = qa_chain(input_text)
chatbot_response

<ipython-input-28-a8dd9566cfe8>:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chatbot_response = qa_chain(input_text)


{'query': '대출과 관련된 정책이 궁금합니다.',
 'result': '서울시에서 운영하는 대출 관련 정책 중 하나는 "서울시 학자금대출 신용회복 지원사업"입니다. 이 정책은 학자금 대출로 인해 신용이 떨어져 어려움을 겪고 있는 청년층을 위한 신용회복 지원을 목적으로 합니다. 지원 대상은 서울에 거주하며 학자금 대출로 인한 신용유의자인 19세부터 39세까지의 청년들입니다. 이 정책은 분할상환약정 체결을 지원하고 초입금을 제공하여 신용유의자 등록을 해제하는 내용을 포함하고 있습니다. 신청은 서울청년포털을 통해 가능하며, 자세한 내용은 관련 사이트를 참고하실 수 있습니다.',
 'source_documents': [Document(metadata={'source': '35.txt'}, page_content='정책내용: 서울시 학자금대출 신용회복 지원사업\n서울특별시 미래청년기획단이 주최하는 금융 정책으로, 학자금 대출로 인해 신용이 떨어져 어려움을 겪고 있는 청년층을 위한 신용회복 지원입니다. 이 정책은 분할상환약정 체결을 지원하고 초입금을 제공함으로써 신용유의자 등록을 해제하는 내용을 포함하고 있습니다. 추가적인 자부담 없이 약정을 체결할 수 있도록 합니다.\n\n지원 대상은 서울에 거주하며 학자금 대출로 인한 신용유의자인 19세부터 39세까지의 청년들이며, 약 200여명을 지원합니다. 2018년부터 2022년에 지원을 받은 사람은 2023년 지원 대상에서 제외됩니다.\n\n신청은 서울청년포털(youth.seoul.go.kr)을 통해 신청할 수 있으며, 심사 및 발표는 매월 1~2회 선정되며, 신청인원에 따라 주기가 변동할 수 있습니다.\n\n이 정책은 서울시 미래청년기획단이 운영하며, 자세한 내용은 관련 사이트를 참고하시기 바랍니다.\n\n[참고사이트]\n신용회복 신청 안내: https://youth.seoul.go.kr/site/main/board/notice/27789?baCategory1=basic&baCommSelec=true\n신청사이트: h

In [ ]:
def get_chatbot_response(chatbot_response):
    print(chatbot_response['result'].strip())
    print('\n문서 출처:')
    for source in chatbot_response["source_documents"]:
        print(source.metadata['source'])

In [ ]:
chatbot_response = qa_chain("신혼 부부의 신혼집 마련을 위한 정책이 있을까?")
get_chatbot_response(chatbot_response)

네, 신혼 부부의 신혼집 마련을 위한 정책이 있습니다. 서울시에서 운영하는 "신혼부부 임차보증금 지원" 정책이 그 예입니다. 이 정책은 주거마련에 대한 부담을 완화하고 더 나은 주거환경을 제공하기 위해 마련되었습니다. 지원 대상은 서울시민이거나 서울로 전입 예정인 신혼부부이며, 혼인신고일 기준으로 7년 이내의 신혼부부 또는 결혼식 예정인 예비신혼부부가 포함됩니다. 대출한도는 임차보증금의 90% 이내 또는 2억원 중 작은 금액이며, 지원 기간은 2023년 1월 1일부터 2023년 12월 31일까지입니다. 자세한 사항은 서울주거포털에서 확인하실 수 있습니다.

문서 출처:
23.txt
23.txt


In [ ]:
chatbot_response = qa_chain("20대를 위한 정책이 있어?")
get_chatbot_response(chatbot_response)

네, 20대를 위한 정책으로는 "2023년 서울시 청년정책 콘테스트 운영"과 "청년 전월세보증보험료 지원"이 있습니다. 두 정책 모두 19세부터 39세까지의 청년들이 참여할 수 있으며, 다양한 지원과 기회를 제공합니다. 자세한 내용은 각각의 정책에 대한 설명을 참고하시면 됩니다.

문서 출처:
8.txt
20.txt


# Gradio 데모

In [35]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 13.1
    Uninstalling websockets-13.1:
      Successfully uninstalled websockets-13.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7


In [ ]:
import gradio as gr

# 인터페이스를 생성.
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="청년정책챗봇") # 청년정책챗봇 레이블을 좌측 상단에 구성
    msg = gr.Textbox(label="질문해주세요!")  # 하단의 채팅창의 레이블
    clear = gr.Button("대화 초기화")  # 대화 초기화 버튼

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
      result = qa_chain(message)
      bot_message = result['result']
      bot_message += ' # sources :'

      # 답변의 출처를 표기
      for i, doc in enumerate(result['source_documents']):
          bot_message += '[' + str(i+1) + '] ' + doc.metadata['source'] + ' '

      # 채팅 기록에 사용자의 메시지와 봇의 응답을 추가.
      chat_history.append((message, bot_message))
      return "", chat_history

    # 사용자의 입력을 제출(submit)하면 respond 함수가 호출.
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    # '초기화' 버튼을 클릭하면 채팅 기록을 초기화.
    clear.click(lambda: None, None, chatbot, queue=False)

# 인터페이스 실행.
demo.launch(debug=True)

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:222: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6666454b7889418800.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
